# PCB Vision AI — Real DeepPCB GPU Training

Run this notebook in Google Colab with **Runtime → Change runtime type → T4 GPU** (or another CUDA GPU). The first code cell verifies TensorFlow can actually see a GPU and stops if it cannot; this notebook never falls back to CPU training.

It clones this project's existing CNN pipeline and the official DeepPCB benchmark. No synthetic data is generated. Dataset download is roughly 1–2 GB and training time depends on the assigned GPU and Colab availability. A Google login is needed for Drive export; Colab may disconnect during long runs. The checkpoint is trained only after you execute the training cell.

In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    raise RuntimeError('No TensorFlow-visible GPU. In Colab select Runtime > Change runtime type > GPU, reconnect, and rerun.')
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as exc:
        raise RuntimeError(f'Could not enable GPU memory growth for {gpu}: {exc}') from exc
print('TensorFlow:', tf.__version__)
print('GPU device(s):', gpus)
print('This notebook will not train on CPU.')

## Fetch existing code and the real dataset

The notebook uses the repository's `PCB-Vision-AI` package and DeepPCB's official annotation and split files.

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/pranavk-10/DL_PCB_Vision.git
%cd /content/DL_PCB_Vision/PCB-Vision-AI
!git clone --depth 1 https://github.com/tangsanli5201/DeepPCB.git data/raw/DeepPCB
PROJECT_DIR = '/content/DL_PCB_Vision/PCB-Vision-AI'
DATASET_DIR = PROJECT_DIR + '/data/raw/DeepPCB'
print('Project:', PROJECT_DIR)
print('DeepPCB:', DATASET_DIR)

In [ ]:
# Keep Colab's preinstalled GPU-enabled TensorFlow; install only supporting packages.
!pip -q install opencv-python-headless scikit-learn matplotlib seaborn
import os, sys, json, numpy as np
sys.path.insert(0, PROJECT_DIR)
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    raise RuntimeError('TensorFlow GPU became unavailable after dependency setup; refusing CPU training.')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
print('Confirmed GPU:', gpus)

## Validate and preprocess DeepPCB

Uses the official 1,000-image `trainval.txt` and 500-image `test.txt`. Validation is held out only from the official trainval set; the official test set remains untouched for final evaluation. All crops from a pair remain in exactly one split.

In [ ]:
from src.dataset_loader import DeepPCBDatasetLoader
from src.preprocessing import PCBDataPipeline
from config.config import DATASET_CONFIG

loader = DeepPCBDatasetLoader(raw_dir=DATASET_DIR)
samples = loader.scan_dataset()
summary = loader.get_dataset_summary(samples)
print(json.dumps(summary, indent=2))
if len(samples) != 1500:
    raise RuntimeError(f'Expected the 1,500 official DeepPCB pairs; found {len(samples)}. Check the dataset clone.')
pipeline = PCBDataPipeline()
pipeline.dataset_config.raw_dir = __import__('pathlib').Path(DATASET_DIR)
data = pipeline.run_full_pipeline(samples=samples, save_to_disk=True)
split = data['split_metadata']
assert len(split['train_samples']) == 850, len(split['train_samples'])
assert len(split['val_samples']) == 150, len(split['val_samples'])
assert len(split['test_samples']) == 500, len(split['test_samples'])
assert set(split['train_samples']).isdisjoint(split['val_samples'])
assert set(split['train_samples']).isdisjoint(split['test_samples'])
assert set(split['val_samples']).isdisjoint(split['test_samples'])
print('Patch counts:', {name: len(data['y_' + name]) for name in ('train', 'val', 'test')})
print('Official image split counts: train=850, validation=150, test=500')

## Train the existing custom CNN on GPU

The existing early-stopping, learning-rate reduction, and best-checkpoint callbacks are used. Training stops if TensorFlow no longer reports a GPU.

In [ ]:
from src.model import build_custom_cnn
from src.train import train_model
from config.config import MODEL_CONFIG, MODELS_DIR, DATASET_CONFIG

if not tf.config.list_physical_devices('GPU'):
    raise RuntimeError('GPU is unavailable; training aborted rather than falling back to CPU.')
tf.keras.utils.set_random_seed(DATASET_CONFIG.random_seed)
checkpoint = MODELS_DIR / 'pcb_cnn_best.keras'
model = build_custom_cnn(
    input_shape=MODEL_CONFIG.input_shape,
    num_classes=MODEL_CONFIG.num_classes,
    dropout_rate=MODEL_CONFIG.dropout_rate,
    learning_rate=MODEL_CONFIG.learning_rate,
)
trained_model, history, elapsed = train_model(
    model, data['X_train'], data['y_train'], data['X_val'], data['y_val'],
    save_path=checkpoint,
    class_weights=data['class_weights'],
    epochs=25,
    batch_size=32,
)
metadata_path = checkpoint.with_suffix('.metadata.json')
metadata = json.loads(metadata_path.read_text())
metadata.update({
    'training_dataset': 'DeepPCB',
    'training_device': [device.name for device in gpus],
    'split_method': 'official DeepPCB trainval/test; validation carved from trainval',
    'train_board_count': len(split['train_samples']),
    'validation_board_count': len(split['val_samples']),
    'test_board_count': len(split['test_samples']),
    'class_weights': {str(k): float(v) for k, v in data['class_weights'].items()},
})
metadata_path.write_text(json.dumps(metadata, indent=2))
print('Checkpoint:', checkpoint)
print('Epochs completed:', len(history['loss']))
print('Best validation accuracy:', max(history['val_accuracy']))
print('Training seconds:', elapsed)

## Held-out evaluation and real-image inference

Reports actual classification metrics from the official test split, reloads the saved checkpoint, and exercises the existing sliding-window localization on one real held-out DeepPCB image.

In [ ]:
from tensorflow.keras.models import load_model
from src.evaluate import evaluate_model_predictions, plot_confusion_matrix
from src.localization import SlidingWindowInspector
from config.config import DATASET_CONFIG, METRICS_DIR, FIGURES_DIR
import cv2

reloaded = load_model(str(checkpoint))
assert tuple(reloaded.input_shape[1:]) == (128, 128, 1)
assert reloaded.output_shape[-1] == len(DATASET_CONFIG.class_labels)
test_probs = reloaded.predict(data['X_test'], batch_size=32, verbose=0)
report = evaluate_model_predictions(data['y_test'], test_probs, DATASET_CONFIG.class_labels)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
(METRICS_DIR / 'test_evaluation.json').write_text(json.dumps(report, indent=2))
plot_confusion_matrix(np.asarray(report['confusion_matrix']), DATASET_CONFIG.class_labels,
                      save_path=FIGURES_DIR / 'confusion_matrix.png')
print(json.dumps(report['overall'], indent=2))
print('Confusion matrix:')
print(np.asarray(report['confusion_matrix']))

test_ids = set(split['test_samples'])
real_test_sample = next(sample for sample in samples if sample.sample_id in test_ids)
real_image = cv2.imread(str(real_test_sample.test_image_path), cv2.IMREAD_COLOR)
if real_image is None:
    raise RuntimeError(f'Could not load held-out image {real_test_sample.test_image_path}')
inspector = SlidingWindowInspector(reloaded)
inspection = inspector.inspect_image(real_image)
print('Real held-out image:', real_test_sample.test_image_path)
print('Windows:', inspection['total_windows_scanned'], 'candidate regions:', len(inspection['detections']))
display(__import__('PIL').Image.fromarray(cv2.cvtColor(inspection['annotated_image'], cv2.COLOR_BGR2RGB)))

## Export checkpoint to Google Drive

After successful training and evaluation, copy both files to Drive. Download them from Drive and place both in the local project's existing `PCB-Vision-AI/models/` folder. Do not rename the metadata independently from the checkpoint. Then start the local dashboard with `python -m streamlit run app.py`; it discovers and loads the checkpoint without retraining.

In [ ]:
from google.colab import drive
import shutil
drive.mount('/content/drive')
export_dir = '/content/drive/MyDrive/PCB-Vision-AI/models'
os.makedirs(export_dir, exist_ok=True)
shutil.copy2(checkpoint, export_dir)
shutil.copy2(metadata_path, export_dir)
print('Exported checkpoint and metadata to:', export_dir)
print('Checkpoint bytes:', os.path.getsize(checkpoint))